# Competition notebook: part-two reproduction (public LB 0.620)

This replaces the v40/v41 stack with the published part-two recipe, which is the strongest
public result that needs no training: **pool_v4 SVC + Qwen3-14B LoRA + Qwen2.5-14B LoRA +
DeBERTa-v3-large**, blended on log probabilities, then Sinkhorn.

| member | trained on | balanced accuracy alone |
|---|---|---|
| n-gram SVC, 4 views | train + 75,783 public passages | 0.473 |
| Qwen3-14B LoRA, 27 of 40 layers | pool, then train | 0.611 |
| Qwen2.5-14B LoRA, 32 of 48 layers | pool, then train | 0.569 |
| DeBERTa-v3-large, full fine tune | pool, then train | 0.546 |
| blend, 5-fold CV / public LB | | 0.642 / 0.620 |

What came out of v40/v41 and why: the final-logit prompt centering, the sibling-pair and
family gates, the orthographic shield and the recall calibration are all gone. The public
part-two notebook lists "gates for one family", "a stacker instead of fixed weights" and
"at most one passage per model inside a topic cluster" as things that were measured and did
not help, and the v41 submission confirmed it for two of them (0.579 against 0.585).

Kept: the centred twin on features (weight 0.1 per LoRA, clusters of 2-18) and Sinkhorn at
300 iterations. New: the end-token fix, which matters because Kaggle's Qwen3-14B-Base ships
the chat tokenizer's end token.

In [ ]:
import gc, glob, json, os, re, shutil, time

# The scoring run has no internet, so transformers must never try to reach the Hub.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import pandas as pd
import torch
from torch import nn

# Keep fp32 sums in fp16 matmuls. A T4 is otherwise allowed to sum in fp16, which drops digits in a deep model.
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False

T0 = time.time()
INPUT_DIR = "/kaggle/input"

# Blend weights, from the public part-two notebook's out-of-fold fit.
W_SVC = 1.0
LORAS = {
    "lora_qwen3_14b":  dict(w=0.4, twin_w=0.1, bs=8, base=dict(model_type="qwen3", hidden_size=5120, num_hidden_layers=40)),
    "lora_qwen25_14b": dict(w=0.2, twin_w=0.1, bs=8, base=dict(model_type="qwen2", hidden_size=5120, num_hidden_layers=48)),
}
W_DEBERTA = 0.25
TWIN = dict(thresh=0.85, min_size=2, max_size=18)

LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
          'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
          'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5',
          'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
K = len(LABELS)
LABEL2I = {l: i for i, l in enumerate(LABELS)}


def find(pattern):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]


def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def vote(name, P, w, rows=slice(None)):
    """Add one member's weighted log probabilities to the running score."""
    LOGIT[rows] += w * np.log(np.clip(P, 1e-9, 1))
    print(f"{name}: weight {w}, done after {time.time() - T0:.0f}s", flush=True)


train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
test["text"] = test["text"].fillna("").astype(str)
y = train["model"].map(LABEL2I).values
LOGIT = np.zeros((len(test), K))
REPORT = {}
print("train", train.shape, "test", test.shape, "gpus", torch.cuda.device_count(), flush=True)

## 1. n-gram SVC on the bigger pool

Four TF-IDF views: raw characters, words with punctuation, word shapes, and small words with
every content word replaced by `CONTENT` so the topic cannot leak. LinearSVC, C 0.5, balanced
class weights, fitted on train plus `pool_v4.parquet` (75,783 passages, up from 34,940).

The block after the vote is a reporting-only 2-fold prompt-grouped out-of-fold SVC on exactly
the production rows. Nothing downstream is tuned on it — it exists so one run says whether the
SVC half of the blend moved. It cannot see the LoRA members out of fold, because their stage two
saw the train passages, so it is an SVC-only number.

In [ ]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

SMALL_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
VIEWS = [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("small", 1, 3, 50000, 0.5)]


def shape_text(t):
    """Every run of letters becomes X or x, so only the rhythm is left."""
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", t)


def small_words_text(t):
    """Keep the small words and the punctuation, and replace every other word by CONTENT."""
    return " ".join(w.lower() if w.lower() in SMALL_WORDS or not w.isalnum() else "CONTENT"
                    for w in re.findall(r"\w+|[^\w\s]", t))


pool = pd.read_parquet(find("pool_v4.parquet"))
pool = pool.dropna(subset=["text", "model"]).copy()
pool["text"] = pool["text"].astype(str)
pool["model"] = pool["model"].astype(str)
pool = pool[pool["model"].isin(LABELS)].copy()
print(f"pool: {len(pool):,} passages, {pool.model.nunique()} models", flush=True)

train_texts = train.text.astype(str).tolist()
test_texts = test.text.astype(str).tolist()
fit_texts = train_texts + pool.text.tolist()
fit_y = np.r_[y, pool.model.map(LABEL2I).values]
N_TR = len(train_texts)

A, B = [], []
for view, lo, hi, max_features, weight in VIEWS:
    prep = shape_text if view == "shape" else small_words_text if view == "small" else (lambda s: s)
    analyzer = "char" if view == "shape" else "word" if view == "small" else view
    opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,
                max_features=max_features, dtype=np.float32)
    if analyzer == "word":
        opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
    vec = TfidfVectorizer(**opts)
    A.append(vec.fit_transform([prep(t) for t in fit_texts]) * weight)
    B.append(vec.transform([prep(t) for t in test_texts]) * weight)
X_FIT = hstack(A, format="csr")
X_TEST = hstack(B, format="csr")
del A, B
gc.collect()

svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight="balanced").fit(X_FIT, fit_y)
vote("svc", softmax(svc.decision_function(X_TEST)), W_SVC)
del svc
gc.collect()

# Reporting only: a 2-fold prompt-grouped out-of-fold SVC on exactly the production rows.
# Nothing downstream is tuned on this; it exists so a run says whether the SVC half of the
# blend moved.  The LoRA members cannot be scored out of fold here (their stage two saw the
# train passages), so this is an SVC-only number, not a full-ensemble one.
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score

if len(train) > 100:
    groups = train["prompt_id"].astype(str).values if "prompt_id" in train else np.arange(N_TR).astype(str)
    pool_rows = np.arange(N_TR, len(fit_texts))
    oof = np.zeros((N_TR, K), np.float32)
    for fi, (itr, iva) in enumerate(GroupKFold(2).split(np.arange(N_TR), y, groups), 1):
        rows = np.r_[itr, pool_rows]
        m = LinearSVC(C=0.5, random_state=100 + fi, max_iter=5000, class_weight="balanced")
        m.fit(X_FIT[rows], fit_y[rows])
        oof[iva] = softmax(m.decision_function(X_FIT[iva]))
        print(f"OOF SVC fold {fi}/2 done ({time.time() - T0:.0f}s)", flush=True)
        del m
        gc.collect()
    REPORT["oof_svc_ba_2fold"] = float(balanced_accuracy_score(y, oof.argmax(1)))
    REPORT["oof_svc_per_class_recall"] = {
        lab: float((oof.argmax(1)[y == i] == i).mean()) for i, lab in enumerate(LABELS)}
    print(f"OOF SVC (2 folds, prompt-grouped) BA={REPORT['oof_svc_ba_2fold']:.5f}", flush=True)
else:
    print("interactive run: skipping the out-of-fold SVC report", flush=True)

del X_FIT
gc.collect()

## 2. LoRA inference

Keep the first layers, merge the adapter into the weights, average the hidden states at 8 depths
without the first token, one LayerNorm per depth, one linear head. Two details that only show up
on Kaggle: the end token must come from the checkpoint's `meta.json` (Kaggle's Qwen3-14B-Base
carries `<|im_end|>`, id 151645, instead of the training token 151643), and a batch that runs out
of memory on the 15 GB T4 is redone one passage at a time.

In [ ]:
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModel, AutoTokenizer


def find_base(spec):
    """The attached base model whose config.json matches spec."""
    for cfg in sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if all(c.get(k) == v for k, v in spec.items()):
            return os.path.dirname(cfg)
    raise FileNotFoundError(f"attach the base model {spec}")


def load_trunk(base_dir, keep, dtype):
    """The first keep decoder layers, split over both T4s when they do not fit on one."""
    mc = AutoConfig.from_pretrained(base_dir)
    mc.num_hidden_layers, mc.use_cache = keep, False
    if getattr(mc, "layer_types", None) is not None:
        mc.layer_types = mc.layer_types[:keep]
    H, I, V = mc.hidden_size, mc.intermediate_size, mc.vocab_size
    hd = getattr(mc, "head_dim", H // mc.num_attention_heads)
    q, kv = mc.num_attention_heads * hd, mc.num_key_value_heads * hd
    gib = 2 * (V * H + keep * (2 * H * q + 2 * H * kv + 3 * H * I)) / 2 ** 30
    if gib > 13 and torch.cuda.device_count() > 1:
        mem = {i: "13GiB" for i in range(torch.cuda.device_count())}
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype, device_map="auto", max_memory=mem).eval()
    else:
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype).to("cuda:0").eval()
    model.norm = nn.Identity()   # the checkpoints were trained without the final norm
    print(f"trunk of {keep} layers, {gib:.1f} GiB of weights", flush=True)
    return model


@torch.no_grad()
def merge_lora(model, adapter_dir):
    """W becomes W + B A alpha / r for every adapted linear layer."""
    acfg = json.load(open(os.path.join(adapter_dir, "adapter_config.json")))
    scale = acfg["lora_alpha"] / acfg["r"]
    sd = load_file(os.path.join(adapter_dir, "adapter_model.safetensors"))
    modules = dict(model.named_modules())
    for k, A in sd.items():
        if ".lora_A." not in k:
            continue
        name = k.split(".lora_A.")[0]
        lin = modules[name[name.index("layers."):]]
        Bm = sd[k.replace(".lora_A.", ".lora_B.")]
        delta = (Bm.float().to(lin.weight.device) @ A.float().to(lin.weight.device)) * scale
        lin.weight.add_(delta.to(lin.weight.dtype))


@torch.no_grad()
def lora_features(ckpt_dir, base_dir, texts, bs):
    """Pooled features Z, one row per passage, and the checkpoint's heads."""
    meta = json.load(open(os.path.join(ckpt_dir, "meta.json")))
    assert meta["labels"] == LABELS, "label order differs from the checkpoint's heads"
    cfg = meta["cfg"]
    dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
    tok = AutoTokenizer.from_pretrained(base_dir)
    tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = load_trunk(base_dir, cfg["keep"], dtype)
    merge_lora(model, os.path.join(ckpt_dir, "adapter"))
    heads = load_file(os.path.join(ckpt_dir, "heads.safetensors"))
    layers, H = cfg["layers"], model.config.hidden_size
    norms = nn.ModuleList([nn.LayerNorm(H) for _ in layers]).to("cuda:0").float()
    norms.load_state_dict({k[len("norms."):]: v for k, v in heads.items() if k.startswith("norms.")})
    ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=cfg["max_len"])["input_ids"]
    # The end token from training, not this tokenizer's. Kaggle's Qwen3-14B-Base carries the
    # chat model's end token (151645 instead of 151643), which silently changes ~14% of its
    # top predictions. The checkpoint's meta.json records the right id.
    eos_id = cfg.get("eos_id", tok.eos_token_id)
    print(f"end token {eos_id}, this tokenizer's is {tok.eos_token_id}", flush=True)
    ids = [x + [eos_id] for x in ids]
    order = np.argsort([len(x) for x in ids], kind="stable")   # similar lengths together means less padding
    Z = np.zeros((len(ids), H * len(layers)), np.float32)

    def embed(idx):
        enc = tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to(model.embed_tokens.weight.device)
        hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
        if not all(torch.isfinite(hs[l]).all() for l in layers):   # fp16 overflow, so redo this batch in bf16
            model.to(torch.bfloat16)
            hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
            model.to(dtype)
        m = enc["attention_mask"].unsqueeze(-1).float().to("cuda:0")
        m[:, 0] = 0   # the first token soaks up attention whatever it is, so leave it out of the average
        den = m.sum(1).clamp(min=1)
        Z[idx] = torch.cat([nm((hs[l].to("cuda:0").float() * m).sum(1) / den) for l, nm in zip(layers, norms)], -1).cpu().numpy()

    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        try:
            embed(idx)
        except torch.cuda.OutOfMemoryError:   # the longest batch can overflow a 15 GB T4
            torch.cuda.empty_cache()
            for i in idx:
                embed([i])
        if s % (bs * 25) == 0:
            print(f"  {s + len(idx)}/{len(ids)} passages ({time.time() - T0:.0f}s)", flush=True)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return Z, {k: v.float().numpy() for k, v in heads.items() if not k.startswith("norms.")}

## 3. Members, with the centred twin

Cluster the test passages by content TF-IDF. In every cluster of 2 to 18 passages, subtract the
cluster mean from the LoRA's features and read the rest with the `centered` head, at weight 0.1.
The twin is not in the CV numbers, because CV has no test clusters.

In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform

if len(test) >= 2:
    Xt = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit_transform(test_texts)
    D = np.clip(1 - (Xt @ Xt.T).toarray(), 0, None)
    np.fill_diagonal(D, 0)
    groups = fcluster(linkage(squareform(D, checks=False), method="average"), TWIN["thresh"], criterion="distance")
    del D, Xt
else:
    groups = np.ones(len(test), dtype=int)
size = np.bincount(groups)[groups]
twin_rows = (size >= TWIN["min_size"]) & (size <= TWIN["max_size"])
print(f"topic clusters: {int((size == 1).sum())} passages alone, {int(twin_rows.sum())} in clusters of 2 to 18", flush=True)

for ckpt_name, mc in LORAS.items():
    ckpt = os.path.dirname(find(f"{ckpt_name}/meta.json"))
    Z, heads = lora_features(ckpt, find_base(mc["base"]), test_texts, mc["bs"])
    vote(ckpt_name, softmax(Z @ heads["plain.weight"].T + heads["plain.bias"]), mc["w"])
    if twin_rows.any():
        Zc = Z.copy()
        for g in np.unique(groups[twin_rows]):
            Zc[groups == g] -= Zc[groups == g].mean(0)
        centred = Zc[twin_rows] @ heads["centered.weight"].T + heads["centered.bias"]
        vote(ckpt_name + " twin", softmax(centred), mc["twin_w"], rows=twin_rows)
    del Z
    gc.collect()

## 4. DeBERTa-v3-large

Fully fine tuned, no LoRA, mean of the last layer, one linear head. The checkpoint is copied to
local disk before loading: reading it straight from the dataset mount took 16 minutes, the copy
plus the load takes 25 seconds.

In [ ]:
ck = os.path.dirname(find("deberta_v3_large/meta.json"))
if ck.startswith("/kaggle/input"):   # one long copy to local disk beats many small reads from the dataset mount
    shutil.copytree(ck, "/tmp/deberta_v3_large", dirs_exist_ok=True)
    ck = "/tmp/deberta_v3_large"
meta = json.load(open(os.path.join(ck, "meta.json")))
assert meta["labels"] == LABELS, "label order differs from the checkpoint's head"
etok = AutoTokenizer.from_pretrained(os.path.join(ck, "encoder"))
enc = AutoModel.from_pretrained(os.path.join(ck, "encoder"), dtype=torch.float32).to("cuda:0").eval()
head = load_file(os.path.join(ck, "head.safetensors"))
W, b = head["weight"].float().numpy(), head["bias"].float().numpy()
ids = etok(test_texts, truncation=True, max_length=meta["max_len"])["input_ids"]
order = np.argsort([len(x) for x in ids], kind="stable")
Z = np.zeros((len(ids), enc.config.hidden_size), np.float32)
with torch.inference_mode():
    for s in range(0, len(order), 32):
        idx = order[s:s + 32]
        bt = etok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to("cuda:0")
        m = bt["attention_mask"].unsqueeze(-1).float()
        Z[idx] = ((enc(**bt).last_hidden_state * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
vote("deberta", softmax(Z @ W.T + b), W_DEBERTA)
del enc, Z
gc.collect()
torch.cuda.empty_cache()

## 5. Blend, Sinkhorn, submit

A weighted sum of log probabilities: SVC 1.0, Qwen3-14B 0.4, Qwen2.5-14B 0.2, DeBERTa 0.25. These
are round numbers chosen on out-of-fold predictions; fitting them finer on 2,699 passages mostly
fits noise. Then Sinkhorn balancing to equal class totals, 300 iterations, skipped on the 3-row
public test.

In [ ]:
def sinkhorn(P, iters=300):
    """One scale per class, so that every class gets an equal share of the probability."""
    c = np.ones(K)
    target = np.full(K, len(P) / K)
    for _ in range(iters):
        R = P * c
        R /= R.sum(1, keepdims=True)
        c *= target / R.sum(0)
    R = P * c
    return R / R.sum(1, keepdims=True)


P = softmax(LOGIT)
if len(test) >= 50:   # skip on the 3 row public test.csv
    P = sinkhorn(P)

sub = pd.DataFrame({"id": test["id"].astype(str), "model": np.array(LABELS)[P.argmax(1)]})
assert len(sub) == len(test) and sub["model"].isin(LABELS).all() and sub["id"].is_unique
sub.to_parquet("submission.parquet", index=False)

REPORT.update({
    "recipe": "public part-two reproduction: pool_v4 SVC + Qwen3-14B LoRA + Qwen2.5-14B LoRA + DeBERTa-v3-large",
    "weights": {"svc": W_SVC, **{k: v["w"] for k, v in LORAS.items()}, "deberta": W_DEBERTA},
    "rows": {"train": int(N_TR), "pool": int(len(pool)), "test": int(len(test_texts))},
    "sinkhorn_iters": 300 if len(test) >= 50 else 0,
    "twin_rows": int(twin_rows.sum()),
})
with open("validation_results.json", "w") as f:
    json.dump(REPORT, f, indent=2)
print(sub["model"].value_counts().to_string())
print(f"done in {time.time() - T0:.0f}s")